# 01 — Real PathSnap dataset exploration

This notebook inspects the organiser-supplied **80 training trips and 10 student-testing trips only**. It does not open the reserved evaluator partition. The archive contains an earlier synthetic demonstration; that demonstration is not evidence of real-data performance.

The production feature extraction and cleaning logic is in [pathsnap_real_training.py](../pathsnap_real_training.py). Data is stored locally under `data/train` and `data/student_test`; it is intentionally excluded from GitHub. If those files are missing, this notebook stops with a clear error rather than inventing data.

Install the repository requirements before executing locally. This notebook is a reproducible analysis view of the real-data pipeline; outputs are generated when run.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'pathsnap_real_training.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook from inside the PathSnap repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Repository:', ROOT)

## 1. File inventory

Each trip has four whitespace-delimited files: `.track` (longitude, latitude, timestamp), `.nodes` (longitude, latitude), `.arcs` (directed node indices), and `.route` (ordered reference arc IDs). The route is not a pointwise GPS label file.

In [ ]:
from pathsnap_real_training import read_table, FEATURES, K, candidate_features

PARTITIONS = {'train': 80, 'student_test': 10}
folders = {}
for split, expected in PARTITIONS.items():
    location = ROOT / 'data' / split
    if not location.is_dir():
        raise FileNotFoundError(f'Missing organiser dataset at {location}')
    trips = sorted(p for p in location.iterdir()
                   if p.is_dir() and (p / (p.name + '.track')).is_file())
    if len(trips) != expected:
        raise ValueError(f'{split}: expected {expected} trips, found {len(trips)}')
    for p in trips:
        for ext in ('.track', '.nodes', '.arcs', '.route'):
            if not (p / (p.name + ext)).is_file():
                raise FileNotFoundError(p / (p.name + ext))
    folders[split] = trips
assert not ({p.name for p in folders['train']} &
            {p.name for p in folders['student_test']})
print({split: len(trips) for split, trips in folders.items()})

## 2. Missing values, timestamp gaps and road-network audit

The following audit is computed from the real files. Non-finite fields and out-of-range coordinates are counted before filtering; time gaps are then measured on chronologically sorted valid observations. Isolated nodes have no incoming or outgoing arcs; this is not a directed reachability test.

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

def load_trip(folder):
    stem = folder.name
    track = read_table(folder / (stem + '.track'), 3)
    nodes = read_table(folder / (stem + '.nodes'), 2)
    arcs = read_table(folder / (stem + '.arcs'), 2, int)
    route = read_table(folder / (stem + '.route'), 1, int).ravel()
    if not np.isfinite(nodes).all():
        raise ValueError(f'Nonfinite road nodes: {folder}')
    if arcs.min() < 0 or arcs.max() >= len(nodes):
        raise ValueError(f'Invalid arc endpoints: {folder}')
    if route.min() < 0 or route.max() >= len(arcs):
        raise ValueError(f'Invalid reference arc: {folder}')
    return track, nodes, arcs, route

def clean_track(track):
    valid = (np.isfinite(track).all(axis=1) &
             (np.abs(track[:, 0]) <= 180) &
             (np.abs(track[:, 1]) <= 90))
    out = track[valid]
    if not len(out):
        raise ValueError('No valid GPS observations')
    return out[np.argsort(out[:, 2], kind='stable')]

audit = []
for split, trips in folders.items():
    for folder in trips:
        raw, nodes, arcs, route = load_trip(folder)
        track = clean_track(raw)
        gaps = np.diff(track[:, 2])
        degree = np.bincount(arcs.ravel(), minlength=len(nodes))
        graph = coo_matrix((np.ones(len(arcs), dtype=np.int32),
                            (arcs[:, 0], arcs[:, 1])),
                           shape=(len(nodes), len(nodes))).tocsr()
        components = connected_components(graph, directed=False,
                                          return_labels=False)
        audit.append(dict(partition=split, trip_id=folder.name,
                          raw_gps=len(raw), valid_gps=len(track),
                          nonfinite_fields=int((~np.isfinite(raw)).sum()),
                          invalid_gps_rows=len(raw)-len(track),
                          nodes=len(nodes), arcs=len(arcs),
                          reference_arcs=len(route),
                          duration_s=float(track[-1, 2]-track[0, 2]),
                          intervals=len(gaps), one_second=int((gaps == 1).sum()),
                          gaps_over_60s=int((gaps > 60).sum()),
                          max_gap_s=float(gaps.max()) if len(gaps) else 0,
                          isolated_nodes=int((degree == 0).sum()),
                          weak_components=components))
audit = pd.DataFrame(audit)
counts = audit.groupby('partition').agg(
    trips=('trip_id', 'size'), gps=('raw_gps', 'sum'),
    valid_gps=('valid_gps', 'sum'),
    invalid_rows=('invalid_gps_rows', 'sum'),
    nonfinite_fields=('nonfinite_fields', 'sum'),
    nodes=('nodes', 'sum'), arcs=('arcs', 'sum'),
    intervals=('intervals', 'sum'), one_second=('one_second', 'sum'),
    gaps_over_60s=('gaps_over_60s', 'sum'),
    isolated_nodes=('isolated_nodes', 'sum'))
counts['one_second_pct'] = 100 * counts.one_second / counts.intervals
display(counts)
display(audit.sort_values('max_gap_s', ascending=False).head(10))

In [ ]:
for field, label in [('raw_gps', 'GPS observations per trip'),
                     ('duration_s', 'Trip duration (seconds)'),
                     ('arcs', 'Road segments per trip'),
                     ('invalid_gps_rows', 'Invalid GPS rows per trip')]:
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for split in PARTITIONS:
        subset = audit.loc[audit.partition == split, field]
        ax.hist(subset, bins=20, alpha=.6, label=split)
    ax.set(xlabel=label, ylabel='Trip count', title='Real development data')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. GPS and road-network visualisation

Select a development trip. The same production candidate extractor used for training identifies nearby arcs. Reference arcs are displayed solely for development analysis, not used in the candidate features.

In [ ]:
from matplotlib.collections import LineCollection
folder = folders['student_test'][0]
raw, nodes, arcs, route = load_trip(folder)
track = clean_track(raw)
X, ids, _, _ = candidate_features(track, nodes, arcs, k=K)
near = np.unique(ids)
ref = np.unique(route)
fig, ax = plt.subplots(figsize=(10, 7))
ax.add_collection(LineCollection(nodes[arcs[near]], colors='grey',
                                 linewidths=.7, alpha=.6, label='Nearby roads'))
ax.add_collection(LineCollection(nodes[arcs[ref]], colors='green',
                                 linewidths=2, label='Reference arcs'))
ax.scatter(track[:, 0], track[:, 1], s=4, color='royalblue', label='GPS')
ax.autoscale()
ax.set(xlabel='Longitude', ylabel='Latitude',
       title=f'Student-testing trip {folder.name}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import folium
m = folium.Map(location=track[:, :2].mean(axis=0)[::-1].tolist(), zoom_start=15)
for name, arc_ids, color, weight in [
    ('Nearby road network', near, 'grey', 2),
    ('Reference route (development only)', ref, 'green', 5)
]:
    layer = folium.FeatureGroup(name=name)
    for arc_id in arc_ids:
        folium.PolyLine(nodes[arcs[arc_id]][:, ::-1].tolist(),
                        color=color, weight=weight).add_to(layer)
    layer.add_to(m)
gps = folium.FeatureGroup(name='Raw GPS')
for lon, lat, _ in track:
    folium.CircleMarker([lat, lon], radius=2, color='blue',
                        fill=True).add_to(gps)
gps.add_to(m)
folium.LayerControl().add_to(m)
m.fit_bounds(track[:, :2][:, ::-1].tolist())
display(m)

## 4. Actual feature schema

The production model uses five road candidates per GPS observation. Each candidate has seven features. The projected distance calculation uses UTM coordinates; road direction, motion and relative position provide additional evidence. Candidate generation is independent of reference routes.

| Feature | Unit or meaning |
|---|---|
| distance_m | Distance to candidate in metres |
| bearing_difference_deg | GPS heading versus road heading |
| speed_mps | Estimated vehicle speed |
| arc_length_m | Segment length in metres |
| projection_fraction | Clipped position along segment (0–1) |
| time_gap_s | Time since previous GPS fix |
| candidate_rank | Distance ordering (0–4) |

In [ ]:
assert X.shape == (len(track), K, len(FEATURES))
assert np.isfinite(X).all()
print('Trip:', folder.name, '| Feature shape:', X.shape)
display(pd.DataFrame(X.reshape(-1, len(FEATURES)), columns=FEATURES).describe())

## 5. Reproduce model preparation

From the repository root, run:

```bash
python -u pathsnap_real_training.py prepare
python -u pathsnap_real_training.py train
```

These commands use only the two development partitions and write local feature caches and model results. Continue with [02_Model_Training.ipynb](02_Model_Training.ipynb) to inspect the already-recorded experiments. Never include the reserved evaluator partition in development.